# Lesson 2: Tool Calling

## Setup

In [1]:
import nest_asyncio
nest_asyncio.apply()

## 1. Define a Simple Tool

In [2]:
! pip install -q llama-index llama-index-llms-anthropic llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.4/198.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.6/82.6 kB 5.6 MB/s eta 0:00:00


In [3]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Soma dois números inteiros."""
    return x + y

def mystery(x: int, y: int) -> int:
    """Função misteriosa que opera em cima de dois números."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [20]:
import os

os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

In [21]:
from llama_index.llms.anthropic import Anthropic

llm = Anthropic(model="claude-3-haiku-20240307")
response = llm.predict_and_call(
    [add_tool, mystery_tool],
    "Diga-me a saída da função misteriosa em 2 e 9",
    verbose=True
)
response.response

=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121


'121'

## 2. Defina uma ferramenta de recuperação automática

### Load Data

Para baixar este artigo, abaixo está o código necessário:

#!wget "https://openreview.net/pdf/474fc6dad3bd9bf7fdb97c7cd72b2cc0649a9647.pdf" -O metagpt.pdf

**Nota**: O arquivo pdf está incluso nesta lição. Para acessá-lo, vá ao menu `Arquivo` e selecione `Abrir...`.

In [22]:
from llama_index.core import SimpleDirectoryReader
# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

In [23]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [ ]:
# print(nodes[0].get_content(metadata_mode="all"))

In [ ]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

In [27]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

In [28]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"}
        ]
    )
)

response = query_engine.query(
    "Quais são alguns resultados de alto nível do MetaGPT?",
)

In [29]:
# print(str(response))
response.response

'Alguns resultados de alto nível do MetaGPT incluem:\n\n- Alcançar um novo estado da arte em benchmarks de geração de código, com 85,9% e 87,7% de taxa de acerto no Pass@1.\n- Superar outros frameworks populares para criação de projetos de software complexos, como AutoGPT, LangChain, AgentVerse e ChatDev, em termos de lidar com níveis mais altos de complexidade de software e oferecer funcionalidade mais extensa.\n- Atingir uma taxa de conclusão de tarefas de 100% em avaliações experimentais, demonstrando a robustez e eficiência (em termos de custo de tempo e tokens) do design do MetaGPT.\n- Utilizar uma abordagem baseada em agentes especializados que seguem padrões estabelecidos, o que permite análise automática de requisitos, design de sistema, geração de código, modificação, execução e depuração durante a execução.'

In [30]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}


### Defina a ferramenta de recuperação automática

In [31]:
from typing import List
from llama_index.core.vector_stores import FilterCondition

def vector_query(
    query: str,
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.

    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.

    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]

    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    response = query_engine.query(query)
    return response


vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [32]:
llm = Anthropic(model="claude-3-haiku-20240307")
response = llm.predict_and_call(
    [vector_query_tool],
    "Quais são os resultados de alto nível do MetaGPT conforme descrito na página 2?",
    verbose=True
)

=== Calling Function ===
Calling function: vector_tool with args: {"query": "MetaGPT results page 2", "page_numbers": []}
=== Function Output ===
Based on the information provided in the context, the key points about MetaGPT's results are:

1. MetaGPT achieves an average score of 3.9 on the SoftwareDev dataset, surpassing the score of 2.1 for ChatDev, which is based on the Chat chain. This shows that MetaGPT outperforms other general intelligent algorithms like AutoGPT, which all scored 1.0 and failed to generate executable code.

2. While models like AutoGPT, Langchain, and AgentVerse display robust general problem-solving capabilities, they lack the ability to systematically deconstruct requirements, which is an essential element for developing complex systems. In contrast, MetaGPT simplifies the process of transforming abstract requirements into detailed class and function designs through its specialized division of labor and SOP workflow.

3. MetaGPT's structured messaging and feed

In [33]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '23', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}
{'page_label': '26', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}


## Vamos adicionar outras ferramentas!

In [34]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "Útil se você deseja obter um resumo do MetaGPT"
    ),
)

In [36]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "Quais são as comparações do MetaGPT com o ChatDev descritas na página 8?",
    verbose=True
)

=== Calling Function ===
Calling function: summary_tool with args: {"input": "Quais s\u00e3o as compara\u00e7\u00f5es do MetaGPT com o ChatDev descritas na p\u00e1gina 8?"}
=== Function Output ===
De acordo com as informações fornecidas, a página 8 descreve as seguintes comparações entre o MetaGPT e o ChatDev:

1. Em relação à executabilidade, o MetaGPT alcança uma pontuação média de 3,9, superando o ChatDev que obteve 2,1. 

2. Comparado a outros algoritmos de inteligência geral, como o AutoGPT, o MetaGPT se destaca, pois esses outros modelos geralmente falham em gerar código executável, com pontuação de 1,0.

3. Enquanto modelos como o AutoGPT, LangChain e AgentVerse demonstram capacidades robustas de resolução de problemas gerais, eles carecem de um elemento essencial para o desenvolvimento de sistemas complexos: a decomposição sistemática de requisitos.

4. Em contraste, o MetaGPT simplifica o processo de transformar requisitos abstratos em designs detalhados de classes e funções p

In [37]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '1', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}
{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}
{'page_label': '3', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}
{'page_label': '3', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}
{'page_label': '4', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2024-11-16', 'last_modified_date': '2024-11-16'}
{'page_lab

In [39]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "O que é um resumo do artigo?",
    verbose=True
)

=== Calling Function ===
Calling function: summary_tool with args: {"input": "O que \u00e9 um resumo do artigo?"}
=== Function Output ===
Infelizmente não posso fornecer um resumo do artigo, pois não tenho acesso ao conteúdo completo do artigo. O artigo fornecido é um preprint sobre um framework de meta-programação chamado MetaGPT, que utiliza procedimentos operacionais padronizados (SOPs) para melhorar a colaboração entre agentes baseados em modelos de linguagem. No entanto, sem ter lido o artigo completo, não posso resumir adequadamente seu conteúdo. Peço desculpas por não poder atender a sua solicitação de resumo neste momento.
